In [1]:
import subprocess
result = subprocess.run([
    "pip", "install", "-q",
    "transformers==4.44.2",
    "peft==0.12.0",
    "accelerate==0.33.0",
    "datasets",
    "scikit-learn",
], capture_output=True, text=True)
print(result.stdout[-3000:] if result.stdout else "")
print(result.stderr[-2000:] if result.stderr else "")

import transformers
print("Transformers version after install:", transformers.__version__)
print("✅ Now RESTART THE KERNEL, then run Cell 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.7 MB/s eta 0:00:00

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
kag

In [2]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Run ONLY after restarting kernel following Cell 1
# ═══════════════════════════════════════════════════════════
import os, json, numpy as np, torch
import transformers
print("Transformers:", transformers.__version__)  # Should be 4.44.2
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

Transformers: 4.44.2
Torch: 2.9.0+cu126
GPU: Tesla T4


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME = "microsoft/deberta-base"
DATA_DIR   = "/kaggle/input/datasets/karthiksekar1821/humaid"
OUTPUT_DIR = "/kaggle/working/deberta_humaid"
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Load Dataset ──────────────────────────────────────────────────────────────
dataset = load_dataset(
    "parquet",
    data_files={
        "train":      f"{DATA_DIR}/train.parquet",
        "validation": f"{DATA_DIR}/val.parquet",
        "test":       f"{DATA_DIR}/test.parquet",
    }
)
print(dataset)

# ── Label Mapping ─────────────────────────────────────────────────────────────
with open(f"{DATA_DIR}/label_mapping.json") as f:
    mapping = json.load(f)

label2id   = mapping["label2id"]
id2label   = {int(k): v for k, v in mapping["id2label"].items()}
num_labels = len(label2id)
print(f"{num_labels} labels:", list(label2id.keys()))

In [ ]:
# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["tweet_text"], truncation=True, max_length=MAX_LEN)

dataset = dataset.map(tokenize, batched=True, remove_columns=["tweet_text"])
dataset = dataset.rename_column("label", "labels")

keep_cols = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in dataset["train"].column_names:
    keep_cols.append("token_type_ids")

dataset.set_format("torch", columns=keep_cols)
print("Columns passed to model:", keep_cols)

# ── Model ─────────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

# ── Data Collator ─────────────────────────────────────────────────────────────
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# ── Training Args ─────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    bf16=False,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
print("\n--- Starting Training ---")
trainer.train()

In [ ]:
# ── Evaluate on Test ──────────────────────────────────────────────────────────
print("\n--- Evaluating on Test Set ---")
test_results = trainer.evaluate(dataset["test"])
print(test_results)

predictions = trainer.predict(dataset["test"])
preds       = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, preds, target_names=list(label2id.keys())))


In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
report = classification_report(
    true_labels, preds,
    target_names=list(label2id.keys()),
    output_dict=True,
)
with open(f"{OUTPUT_DIR}/test_results.json", "w") as f:
    json.dump({"overall": test_results, "per_class": report}, f, indent=4)

trainer.save_model(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")
print(f"\nDone. Results + model saved to {OUTPUT_DIR}")